## 4 Training BERT

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import tensorflow as tf
import datetime

import datasets
from datasets import load_dataset

/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset, DatasetDict

# Carica il dataset
dataset = load_dataset('csv', data_files='cleaned_stance_dataset_enriched.csv', encoding = "utf-8", sep=',')
#
dataset = dataset.filter(lambda example: example['stance'] is not None)
dataset

DatasetDict({
    train: Dataset({
        features: ['folder', 'pulsar_file', 'id', 'search', 'content id', 'source', 'application', 'title', 'content', 'date', 'parent', 'language', 'url', 'parent source identifier', 'domain', 'credibility', 'credibility score', 'topics', 'image tags', 'tags', 'sentiment', 'sentiment class', 'sentiment by', 'main emotion', 'visibility', 'ave', 'duration', 'circulation', 'media reach', 'media impressions', 'social impressions', 'city', 'country', 'region', 'latitude', 'longitude', 'user city', 'user country', 'user latitude', 'user longitude', 'no. of followers', 'no. of friends', 'gender', 'bio', 'user views', 'Job titles', 'Companies', 'Industries', 'links url', 'no. of comments', 'no. of likes', 'no. of shares', 'no. of views', 'Linkedin reactions', 'Engagements/social shares', 'user name', 'user screen name', 'user source id', 'eRep', 'CSR', 'trust', 'CX', 'intensity', 'climate change', 'environmental impact', 'biodiversity', 'human rights', 'labo

In [4]:
# df_raw = pd.read_csv('merged_data_pulsar_groundtruth_add_features.csv')
# dataset=df_raw.copy()

# stance_mapping = {
#     0: "NOT DEFINED",
#     1: "Promotional",
#     2: "Neutral",
#     3: "Discouraging",
#     4: "Ambiguous",
#     5: "indefinite/ironic",
# }
# # loading dataframe
# df_raw['source'] = df_raw['source'].astype(str)
# # to not consider samples with NOT DEFINED or indefinite/ironic class
# df_raw = df_raw[~df_raw.stance.isin([0, 4,5])]
# # assigning a classname
# df_raw['stance_name'] = df_raw['stance'].apply(lambda x: stance_mapping[x])
# df_raw.head()
# # selecting only tweets where the 'post subtype' is 'original post' and facebook entries
# dataset = df_raw[df_raw['source'].str.contains("Facebook Pages") | (df_raw['source'].str.contains("X") & df_raw['post subtype'].str.contains("original post"))]
# # from here you can start working

### Trasnform stance from 1,2,3 00> 0,1,2

In [5]:
# def transform_stance(example):
#     example['stance'] = example['stance'] - 1
#     return example

# dataset = dataset.map(transform_stance)

# dataset

In [6]:
dataset = dataset.cast_column('stance', datasets.ClassLabel(num_classes=5, names=[0,1,2,3,4]))

# Divide in training e test+validation, stratificando su 'Stance'
train_testvalid = dataset['train'].train_test_split(test_size=0.2, stratify_by_column='stance')

# Divide test+validation in test e validation, stratificando su 'Stance'
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, stratify_by_column='stance')

# Crea un nuovo DatasetDict con i tre set
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']
})

In [7]:
# Estrae il dataset dal dizionario
df_train = dataset['train']
df_valid = dataset['valid']
df_test = dataset['test']

In [8]:
from transformers import AutoTokenizer

#model_name ="albert-base-v2"
model_name = "distilbert/distilbert-base-uncased"
#model_name = "FacebookAI/xlm-roberta-base"
#model_name = "dbmdz/bert-base-italian-uncased"
#model_name ="google-bert/bert-base-uncased"
#model_name ="RoBERTa"

tokenizer = AutoTokenizer.from_pretrained(model_name, 
                                          force_download=False,
                                          cache_dir="./cache_huggingface")


def tokenize_function(examples):
    texts = examples["content"]
    examples['labels'] = [label for label in examples['stance']]
    #examples['Stance'] = [int(label) if label is not None else -1 for label in examples['Stance']]
    texts = [str(text) for text in texts]
    return tokenizer(texts, padding="max_length", truncation=True)

In [9]:
tokenized_datasets = df_train.map(
    tokenize_function,
    batched=True,
    #remove_columns = ['Unnamed: 0', 'data', 'link', 'content', 'researcher', 'INCLUSIONE', "user's profession", 'Visibility', 'Potential impressions', 'Actual impressions', 'No. Of comments', 'No. Of likes', 'No. Of shares', 'No. Of retweets', 'No. Of views', 'Narrative stance', 'Conspiracy theories', 'False ingredients and components', 'safety concerns', 'Efficacy concerns', 'Natural immunity superiority', 'Big Pharma', 'Distrust in health atuthorities', 'Religious belief', 'narrative distorte ricorrenti', 'Narrativa: note', 'Information Source', 'Topic', 'Personal storytelling about vaccine', 'Tone of voice'],
)

tokenized_datasets_valid = df_valid.map(
    tokenize_function,
    batched=True,
    #remove_columns = ['Unnamed: 0', 'data', 'link', 'content', 'researcher', 'INCLUSIONE', "user's profession", 'Visibility', 'Potential impressions', 'Actual impressions', 'No. Of comments', 'No. Of likes', 'No. Of shares', 'No. Of retweets', 'No. Of views', 'Narrative stance', 'Conspiracy theories', 'False ingredients and components', 'safety concerns', 'Efficacy concerns', 'Natural immunity superiority', 'Big Pharma', 'Distrust in health atuthorities', 'Religious belief', 'narrative distorte ricorrenti', 'Narrativa: note', 'Information Source', 'Topic', 'Personal storytelling about vaccine', 'Tone of voice'],
)

Map: 100%|██████████| 222/222 [00:00<00:00, 3780.97 examples/s]


In [10]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

#model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=5)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import torch
print(torch.backends.mps.is_available())  # True se MPS è supportato

True


In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10  ,
    save_steps=1000,
    save_total_limit=2,
)

# training_args = TrainingArguments(
#     output_dir="bert-base-uncased",
#     learning_rate=2e-5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=5,
#     weight_decay=0,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True
# )

/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [13]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [14]:
import os
import torch

mps = True #Silicon M1 hardware

if mps:
    torch.mps.empty_cache()
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
else:
    torch.cuda.empty_cache()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Inizializza il Trainer
trainer = Trainer(
    model=model.to(device),
    args=training_args,
    train_dataset=tokenized_datasets,
    eval_dataset=tokenized_datasets_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


/var/folders/qq/95qwkbxs3zb9bw0w7n7tysmr0000gp/T/ipykernel_51358/2177400404.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
# Configura i parametri di addestramento
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    save_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model.to(device),
    args=training_args,
    train_dataset=tokenized_datasets,
    eval_dataset=tokenized_datasets_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
# Inizializza il Trainer
# trainer = Trainer(
#     model=model.to(device),
#     args=training_args,
#     train_dataset=tokenized_datasets,
#     eval_dataset=tokenized_datasets_valid,
#     tokenizer=tokenizer,
#     data_collator=data_collator,
# )



/var/folders/qq/95qwkbxs3zb9bw0w7n7tysmr0000gp/T/ipykernel_51358/2510830474.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
learning = True
resume_from_checkpoint = True

import os
import re

def get_latest_checkpoint(path='results'):
    checkpoints = []
    for name in os.listdir(path):
        match = re.match(r'checkpoint-(\d+)', name)
        if match:
            checkpoints.append((int(match.group(1)), name))
    if not checkpoints:
        return None
    latest = max(checkpoints)[1]
    return os.path.join(path, latest)


if learning:
    if resume_from_checkpoint:
        latest_checkpoint=get_latest_checkpoint("results")
        trainer.train(latest_checkpoint)
    else:
        trainer.train()
    trainer.save_model("./BERT")

Epoch,Training Loss,Validation Loss
1,No log,1.058461
2,1.085500,1.058495
3,1.069700,1.061803
4,1.058700,1.056489
5,1.057500,1.048637


In [19]:
path = './BERT'
model = AutoModelForSequenceClassification.from_pretrained(path)
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [20]:
new_df = df_test.select_columns(['content', 'stance']).to_pandas()
import torch

predictions = []
test = []
softmax_list = []
for i, row in new_df.iterrows():
  text = row["content"]
  test.append(row["stance"])
  inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)
  # Remove token_type_ids if present

  if "token_type_ids" in inputs:
      inputs.pop("token_type_ids")


  with torch.no_grad():
    outputs = model(**inputs).logits
  paraphrased_text = torch.softmax(outputs, dim=1).tolist()[0]
  softmax_list.append(paraphrased_text)

In [21]:
for elem in softmax_list:
  predictions.append(elem.index(max(elem)) + 1)

In [23]:
from sklearn.metrics import classification_report

# Define target names for all classes
#target_names = ['0', '1', '2', '3', '4']
target_names = [ '1', '2', '3']

# Ensure alignment of labels and predictions
print("Unique classes in test:", set(test))
print("Unique classes in predictions:", set(predictions))

# Generate the classification report
print(classification_report(test, predictions, target_names=target_names))

Unique classes in test: {1, 2, 3}
Unique classes in predictions: {2}
              precision    recall  f1-score   support

           1       0.00      0.00      0.00       105
           2       0.31      1.00      0.47        69
           3       0.00      0.00      0.00        49

    accuracy                           0.31       223
   macro avg       0.10      0.33      0.16       223
weighted avg       0.10      0.31      0.15       223



/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _wa

In [24]:
from sklearn.metrics import classification_report
target_names = ['1', '2', '3']
print(classification_report(test, predictions, target_names = target_names))

              precision    recall  f1-score   support

           1       0.00      0.00      0.00       105
           2       0.31      1.00      0.47        69
           3       0.00      0.00      0.00        49

    accuracy                           0.31       223
   macro avg       0.10      0.33      0.16       223
weighted avg       0.10      0.31      0.15       223



/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _wa

In [25]:
from sklearn.metrics import classification_report
target_names = ['1', '2', '3']
print(classification_report(test, predictions, target_names = target_names))

              precision    recall  f1-score   support

           1       0.00      0.00      0.00       105
           2       0.31      1.00      0.47        69
           3       0.00      0.00      0.00        49

    accuracy                           0.31       223
   macro avg       0.10      0.33      0.16       223
weighted avg       0.10      0.31      0.15       223



/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/sergioparigi/Desktop/TESI/Tesi Sergio/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _wa